# 1. What you'll learn

        - diagnose trend with ADF and differencing
- fit ARIMA and ETS on the same history
- backtest rolling origins with a seasonal-naive baseline

        **The one question this notebook answers:** How do differencing, autocorrelation, and component smoothing produce forecasts and intervals?

# 2. Setup

This lesson keeps arima and exponential smoothing small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Twelve years of monthly demand combine trend, annual seasonality, AR noise, and a late level shift. The last 18 months are held out chronologically.

**Small example:** For levels [10,12,15,16,18], first differences are [2,3,1,2]. ARIMA models the more stable changes, then cumulatively adds forecasts back to the level.

In [ ]:
rng=np.random.default_rng(SEED);n=144;t=np.arange(n);noise=np.zeros(n)
for i in range(1,n): noise[i]=.55*noise[i-1]+rng.normal(0,1.5)
y=50+.22*t+7*np.sin(2*np.pi*t/12)+np.where(t>=105,5,0)+noise;series=pd.Series(y,index=pd.date_range("2014-01-01",periods=n,freq="MS"));print("Shape:",series.shape,"last value:",round(series.iloc[-1],2))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
diff=series.diff().dropna();fig,axes=plt.subplots(1,3,figsize=(14,3.7));series.plot(ax=axes[0]);axes[0].set_title("Nonstationary level");diff.plot(ax=axes[1],color="#176b66");axes[1].set_title("First difference");plot_acf(diff,lags=24,ax=axes[2]);axes[2].set_title("ACF of differences");plt.tight_layout();plt.show();print("ADF p level/difference:",round(adfuller(series)[1],3),round(adfuller(diff)[1],3))

**Takeaway:** Trend and a level shift make the raw level nonstationary.

**Takeaway:** Differencing removes most long-run level movement.

**Takeaway:** Seasonal and short-lag correlation remain after differencing and can be modelled.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
train=series.iloc[:-18];test=series.iloc[-18:];print("Chronological train/test:",train.shape,test.shape,"cutoff:",train.index[-1].date())

# 6. Train

        Training turns the assumptions behind arima and exponential smoothing into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Difference the level to represent stable changes.
2. Fit AR and MA dependence plus a seasonal term by likelihood.
3. ETS recursively updates level, trend, and seasonal states.
4. Forecast from the final state and compare at rolling historical origins.

In [ ]:
arima=SARIMAX(train,order=(1,1,1),seasonal_order=(1,0,0,12),trend="t",enforce_stationarity=False,random_state=SEED).fit(disp=False,maxiter=80);ets=ExponentialSmoothing(train,trend="add",seasonal="add",seasonal_periods=12,initialization_method="estimated").fit(optimized=True);print("ARIMA AIC:",round(arima.aic,1),"ETS SSE:",round(ets.sse,1))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
af=arima.get_forecast(18);ap=af.predicted_mean;ci=af.conf_int();ep=ets.forecast(18);sn=pd.Series(np.resize(train.iloc[-12:].to_numpy(),len(test)),index=test.index)
scores={"seasonal naive":mean_absolute_error(test,sn),"ARIMA":mean_absolute_error(test,ap),"ETS":mean_absolute_error(test,ep)};print("MAE:",{k:round(v,3) for k,v in scores.items()})
fig,axes=plt.subplots(1,3,figsize=(14,3.7));series.iloc[-42:].plot(ax=axes[0],label="actual");ap.plot(ax=axes[0],label="ARIMA");ep.plot(ax=axes[0],label="ETS");axes[0].fill_between(test.index,ci.iloc[:,0],ci.iloc[:,1],alpha=.18);axes[0].legend();axes[0].set_title("Forecast and ARIMA interval")
axes[1].bar(scores.keys(),scores.values(),color=[".6","#176b66","#d38b45"]);axes[1].set_title("Chronological MAE")
axes[2].plot(test.index,test-ap,marker="o",label="ARIMA");axes[2].plot(test.index,test-ep,marker="o",label="ETS");axes[2].axhline(0,ls="--",color="black");axes[2].set_title("Forecast errors");axes[2].legend();plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Ignoring trend/seasonality violates stationary-residual assumptions and produces biased forecasts or misleading intervals after shifts. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
wrong=SARIMAX(train,order=(1,0,1),seasonal_order=(0,0,0,0),trend="c",enforce_stationarity=False).fit(disp=False,maxiter=60);wp=wrong.forecast(18);coverage=np.mean((test>=ci.iloc[:,0])&(test<=ci.iloc[:,1]));print(f"Differenced seasonal ARIMA MAE={scores['ARIMA']:.3f}; undifferenced nonseasonal MAE={mean_absolute_error(test,wp):.3f}; nominal 95% interval coverage={coverage:.1%}")

**Use this when…** one series has stable autocorrelation or evolving trend/seasonal states and honest rolling backtests.

        **Don't use this when…** many series should share strength, drivers matter but are omitted, or structural breaks dominate history.

        ## Try this

        1. Remove the level shift and compare interval coverage.
2. Run six rolling origins and plot score variability.
3. Add a known external driver through SARIMAX exog.